# ASR модель — приватный репозиторий

В Colab открой Secrets (значок ключа), добавь `GITHUB_TOKEN` и включи Notebook access.
У токена должен быть доступ к `HW1_ASR` с разрешением Contents: Read-only.

Вставь ссылку на датасет ниже и запускай ячейки по порядку. Ноутбук скачивает ветку `main`; локальные изменения сначала нужно закоммитить и отправить в GitHub.

Модель: exp107, CNN + BiGRU + CTC. Декодер: beam 32 + word LM, alpha=0.6, word_bonus=2.

In [ ]:
dataset_url = ""

## 1. Установка

Клонируем приватный репозиторий или обновляем существующую копию через `git pull --ff-only`. Токен передаётся только процессу Git, без записи в URL репозитория.

Создаём отдельный Python 3.11, если окружения ещё нет, и устанавливаем зависимости из обновлённого requirements.txt.

In [ ]:
import os
import sys
import subprocess
import tempfile
from pathlib import Path
from google.colab import userdata
os.environ["MPLBACKEND"] = "Agg"
project = Path("/content/HW1_ASR")
git_env = os.environ.copy()
git_env["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN").strip()
git_env["GIT_TERMINAL_PROMPT"] = "0"
try:
    with tempfile.TemporaryDirectory() as temporary:
        askpass = Path(temporary) / "askpass.sh"
        askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) printf \'%s\\n\' \'ignisiris\' ;;\n  *Password*) printf \'%s\\n\' "$GITHUB_TOKEN" ;;\nesac\n')
        askpass.chmod(0o700)
        git_env["GIT_ASKPASS"] = str(askpass)
        if project.exists():
            command = ["git", "-c", "credential.helper=", "-C", str(project), "pull", "--ff-only", "origin", "main"]
        else:
            command = ["git", "-c", "credential.helper=", "clone", "--branch", "main", "https://github.com/ignisiris/HW1_ASR.git", str(project)]
        subprocess.run(command, env=git_env, check=True)
finally:
    git_env.pop("GITHUB_TOKEN", None)
os.chdir(project)
python = "/content/asr-env/bin/python"
if not Path(python).exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "uv"], check=True)
    subprocess.run([sys.executable, "-m", "uv", "venv", "--python", "3.11", "--seed", "/content/asr-env"], check=True)
subprocess.run([python, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
subprocess.run([python, "-c", "import inference; print('Dependencies OK')"], check=True)

## 2. Ресурсы модели

Скрипт скачивает веса, языковую модель и логи.

In [ ]:
subprocess.run([python, "download_resources.py"], check=True)
model_config = "experiments/inference_exp107_lm"

## 3. Инференс и метрики

Вызываем inference.py, затем calc_metrics.py.

In [ ]:
run_dir = Path("/content/inference")
run_dir.mkdir(exist_ok=True)
def recognize(data_dir, name):
    output = run_dir / name
    subprocess.run([python, "inference.py", "-cn", model_config, f"datasets.custom.data_dir={data_dir}", f"inferencer.save_path={output}"], check=True)
    predictions = output / "custom"
    references = data_dir / "transcriptions"
    if references.is_dir():
        result = subprocess.run([python, "calc_metrics.py", str(references), str(predictions)], stdout=subprocess.PIPE, text=True, check=True)
        print("Итоговые метрики по всему датасету:")
        print(result.stdout)
    print("Предсказания:", predictions)
    return predictions

## 4. Готовые примеры

Шесть записей: три dev-clean и три dev-other.

In [ ]:
sample_dir = project / "demo/sample"
recognize(sample_dir, "sample")

## 5. Датасет по ссылке

In [ ]:
input_dir = run_dir / "dataset"
subprocess.run([python, "prepare_demo_dataset.py", dataset_url.strip(), "--output", str(input_dir)], check=True)
recognize(input_dir, "user")